# 01 · Preparación de datos

**Fase del plan:** F1 – Preparación de datos.

**Objetivo:** construir el panel de países (237 Country/Area, 1950–2100) con los indicadores de `IND_AMPLIADO`, asignar subregión y región geográfica, marcar los países pequeños, transformar CNMR según `TRANSF_CNMR` y guardar las matrices país × año que usarán las fases siguientes.

**Entradas:** `data/WPP2024_Demographic_Indicators_Medium_csv.gz`, `data/WPP2024_Supplement_Births_and_LE60.csv`, `output/data/config.rds`.

**Salidas:**
- `output/data/panel_paises.rds` (formato largo)
- `output/data/meta_paises.rds` (una fila por país)
- `output/data/matrices_1950_2023.rds` y `output/data/matrices_1950_2100.rds` (lista de matrices país × año por indicador)


## Configuración

Se cargan los paquetes, la configuración común (`config.rds`) y la semilla.


In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(here)
})
cfg <- readRDS(here::here("output/data/config.rds")); invisible(list2env(cfg, envir = globalenv())); set.seed(SEED)
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 120, width = 150)
cat("Indicadores ampliados:", IND_AMPLIADO, "| TRANSF_CNMR:", TRANSF_CNMR, "\n")


Indicadores ampliados: TFR LEx CNMR MAC MedianAgePop | TRANSF_CNMR: asinh 


## a) Lectura de los datos

Se lee el archivo principal (variante Medium, 1950–2101) con `readr::read_csv()`, que maneja el BOM de la primera columna, y el suplemento de nacimientos por sexo y LE60. Se verifica que el archivo principal tiene 84,360 filas (555 LocID × 152 años) y que el nombre de la primera columna quedó limpio (`SortOrder`, sin BOM).


In [2]:
medium <- readr::read_csv(RUTA_MEDIUM, show_col_types = FALSE, guess_max = 1e5)
suplem <- readr::read_csv(RUTA_SUPLEM, show_col_types = FALSE, guess_max = 1e5)

cat("Medium: ", dim(medium), "\n"); print(names(medium))
cat("\nSupplement: ", dim(suplem), "\n"); print(names(suplem))

stopifnot(nrow(medium) == 84360,
          n_distinct(medium$LocID) == 555,
          n_distinct(medium$Time) == 152,
          !any(grepl("﻿", names(medium))),
          all(c("LocID", "Location", "ISO3_code", "LocTypeName", "ParentID", "Time",
                IND_AMPLIADO, "TPopulation1July") %in% names(medium)))


Medium:  84360 67 
 [1] "SortOrder"              "LocID"                  "Notes"                  "ISO3_code"              "ISO2_code"             
 [6] "SDMX_code"              "LocTypeID"              "LocTypeName"            "ParentID"               "Location"              
[11] "VarID"                  "Variant"                "Time"                   "TPopulation1Jan"        "TPopulation1July"      
[16] "TPopulationMale1July"   "TPopulationFemale1July" "PopDensity"             "PopSexRatio"            "MedianAgePop"          
[21] "NatChange"              "NatChangeRT"            "PopChange"              "PopGrowthRate"          "DoublingTime"          
[26] "Births"                 "Births1519"             "CBR"                    "TFR"                    "NRR"                   
[31] "MAC"                    "SRB"                    "Deaths"                 "DeathsMale"             "DeathsFemale"          
[36] "CDR"                    "LEx"                    "LExMale"       

## b) Ubicaciones y jerarquía geográfica

Se extrae una tabla con una fila por `LocID`. La jerarquía geográfica se deriva de `ParentID`:

- **País → subregión:** el `ParentID` de un país es su subregión.
- **Subregión → región:** el `ParentID` de la subregión es su región geográfica (903 Africa, 935 Asia, 908 Europe, 904 Latin America and the Caribbean, 909 Oceania).
- **Excepción Northern America:** los países con `ParentID` 918 (Northern America) no tienen una fila propia para la subregión 918, así que se asignan a mano a la región 905 "Northern America"; su subregión también es Northern America.

Las uniones se hacen siempre por `LocID` (hay nombres de `Location` duplicados). Checks: los 237 países tienen región y hay exactamente 6 regiones.


In [3]:
ubic <- medium |>
  distinct(LocID, Location, ISO3_code, LocTypeName, ParentID)
stopifnot(nrow(ubic) == n_distinct(medium$LocID))   # una fila por LocID

nombres <- ubic |> select(LocID, Location)
REGIONES_GEO <- c(903, 935, 908, 904, 909, 905)

paises_geo <- ubic |>
  filter(LocTypeName == "Country/Area") |>
  select(LocID, Location, ISO3_code, subregion_id = ParentID) |>
  left_join(ubic |> select(subregion_id = LocID, region_id = ParentID), by = "subregion_id") |>
  mutate(
    # Northern America (918) no tiene filas propias → región 905
    region_id    = if_else(subregion_id == 918, 905, region_id),
    subregion    = if_else(subregion_id == 918, "Northern America",
                           nombres$Location[match(subregion_id, nombres$LocID)]),
    region       = nombres$Location[match(region_id, nombres$LocID)]
  )

stopifnot(nrow(paises_geo) == 237,
          !anyNA(paises_geo$region), !anyNA(paises_geo$subregion),
          n_distinct(paises_geo$region_id) == 6,
          all(paises_geo$region_id %in% REGIONES_GEO),
          !anyNA(paises_geo$ISO3_code), !anyDuplicated(paises_geo$ISO3_code))

paises_geo |> count(region_id, region, name = "n_paises") |> print()
cat("\nSubregiones:", n_distinct(paises_geo$subregion_id), "\n")
paises_geo |> filter(subregion_id == 918) |> select(LocID, Location, subregion, region) |> print()


# A tibble: 6 × 3
  region_id region                          n_paises
      <dbl> <chr>                              <int>
1       903 Africa                                58
2       904 Latin America and the Caribbean       50
3       905 Northern America                       5
4       908 Europe                                50
5       909 Oceania                               23
6       935 Asia                                  51

Subregiones: 22 
# A tibble: 5 × 4
  LocID Location                  subregion        region          
  <dbl> <chr>                     <chr>            <chr>           
1    60 Bermuda                   Northern America Northern America
2   124 Canada                    Northern America Northern America
3   304 Greenland                 Northern America Northern America
4   666 Saint Pierre and Miquelon Northern America Northern America
5   840 United States of America  Northern America Northern America


## c) Filtro de países y horizonte

Se conservan solo las filas con `LocTypeName == "Country/Area"` y `Time` entre 1950 y 2100. El año 2101 se descarta porque solo trae `TPopulation1Jan`. Se añaden las variables geográficas uniendo por `LocID`.

Checks: 237 `LocID` únicos, 151 años por país y ningún NA en los indicadores de `IND_AMPLIADO` ni en `TPopulation1July`.


In [4]:
panel <- medium |>
  filter(LocTypeName == "Country/Area", Time >= HORIZONTE_SENS[1], Time <= HORIZONTE_SENS[2]) |>
  select(LocID, Time, all_of(IND_AMPLIADO), TPopulation1July) |>
  inner_join(paises_geo |> select(LocID, ISO3_code, Location, region, subregion), by = "LocID") |>
  arrange(LocID, Time)

anios_por_pais <- panel |> count(LocID)
na_por_var <- colSums(is.na(panel[, c(IND_AMPLIADO, "TPopulation1July")]))

stopifnot(n_distinct(panel$LocID) == 237,
          all(anios_por_pais$n == 151),
          !anyDuplicated(panel[, c("LocID", "Time")]),
          all(na_por_var == 0))

cat("Dimensiones del panel:", dim(panel), "\n")
cat("Años:", range(panel$Time), "| años por país:", unique(anios_por_pais$n), "\n")
print(na_por_var)


Dimensiones del panel: 35787 12 
Años: 1950 2100 | años por país: 151 
             TFR              LEx             CNMR              MAC     MedianAgePop TPopulation1July 
               0                0                0                0                0                0 


## d) Consistencia con el total mundial

La población de los 237 países debe sumar la de World (`LocID` 900) en cada año:

$$
\max_t \Big|\sum_{i=1}^{237} \mathrm{TPopulation1July}_i(t) - \mathrm{TPopulation1July}_{\text{World}}(t)\Big| < 0.1 \text{ (miles)}.
$$

Así se confirma que el filtro no omite ni duplica países.


In [5]:
consistencia <- panel |>
  group_by(Time) |>
  summarise(suma_paises = sum(TPopulation1July), .groups = "drop") |>
  inner_join(medium |> filter(LocID == 900) |> select(Time, world = TPopulation1July), by = "Time") |>
  mutate(dif = suma_paises - world)

stopifnot(nrow(consistencia) == 151)
dif_max <- max(abs(consistencia$dif))
cat(sprintf("Diferencia absoluta máxima (miles): %.4f en el año %d\n",
            dif_max, consistencia$Time[which.max(abs(consistencia$dif))]))
stopifnot(dif_max < 0.1)


Diferencia absoluta máxima (miles): 0.0140 en el año 2024


## e) Países pequeños

Se marca `es_pequeno = TRUE` cuando la población a mitad de año en `ANIO_REF_POP` (2023) es menor que `MIN_POP_MILES` (100 mil habitantes). En estos países, unos pocos migrantes mueven mucho las tasas por 1,000 (sobre todo CNMR), así que se excluyen del análisis principal y solo entran en la sensibilidad (F8). Deben ser 37.


In [6]:
pop_ref <- panel |>
  filter(Time == ANIO_REF_POP) |>
  transmute(LocID, pop_2023 = TPopulation1July, es_pequeno = TPopulation1July < MIN_POP_MILES)

n_pequenos <- sum(pop_ref$es_pequeno)
cat("Países pequeños (<", MIN_POP_MILES, "mil en", ANIO_REF_POP, "):", n_pequenos, "\n\n")
stopifnot(nrow(pop_ref) == 237, n_pequenos == 37)

pop_ref |>
  filter(es_pequeno) |>
  inner_join(paises_geo |> select(LocID, ISO3_code, Location, region), by = "LocID") |>
  arrange(pop_2023) |>
  transmute(ISO3_code, Location, region, pop_2023 = round(pop_2023, 2)) |>
  as.data.frame() |> print()


Países pequeños (< 100 mil en 2023 ): 37 

   ISO3_code                         Location                          region pop_2023
1        VAT                         Holy See                          Europe     0.50
2        NIU                             Niue                         Oceania     1.82
3        TKL                          Tokelau                         Oceania     2.40
4        FLK      Falkland Islands (Malvinas) Latin America and the Caribbean     3.48
5        MSR                       Montserrat Latin America and the Caribbean     4.42
6        SHN                     Saint Helena                          Africa     5.29
7        SPM        Saint Pierre and Miquelon                Northern America     5.68
8        TUV                           Tuvalu                         Oceania     9.82
9        BLM                 Saint Barthélemy Latin America and the Caribbean    11.09
10       WLF        Wallis and Futuna Islands                         Oceania    11.37


## f) Transformación de CNMR

La tasa neta de migración (CNMR, por 1,000) tiene picos extremos por crisis, guerras y países pequeños, que podrían dominar la varianza y generar grupos espurios. `CNMR_t` se calcula según `TRANSF_CNMR`:

- `"asinh"` (principal): $\mathrm{CNMR}_t = \operatorname{asinh}(\mathrm{CNMR}) = \log\!\big(\mathrm{CNMR} + \sqrt{\mathrm{CNMR}^2 + 1}\big)$. Se comporta como la identidad cerca de 0 y como un logaritmo con signo en las colas, así que conserva el signo (emigración vs. inmigración) y comprime los extremos.
- `"winsor"`: se recorta CNMR a los percentiles $[q_{0.01}, q_{0.99}]$, calculados sobre todos los países y años del horizonte principal (`HORIZONTE`, 1950–2023). Los mismos cortes se aplican a los años de proyección.
- `"ninguna"`: $\mathrm{CNMR}_t = \mathrm{CNMR}$.


In [7]:
transformar_cnmr <- function(x, metodo, cortes = NULL) {
  switch(metodo,
         asinh   = asinh(x),
         winsor  = pmin(pmax(x, cortes[1]), cortes[2]),
         ninguna = x,
         stop("TRANSF_CNMR no reconocido: ", metodo))
}

cortes_cnmr <- panel |>
  filter(Time >= HORIZONTE[1], Time <= HORIZONTE[2]) |>
  pull(CNMR) |>
  quantile(c(0.01, 0.99), names = FALSE)

panel <- panel |> mutate(CNMR_t = transformar_cnmr(CNMR, TRANSF_CNMR, cortes_cnmr))

stopifnot(!anyNA(panel$CNMR_t),
          all(sign(panel$CNMR_t) == sign(panel$CNMR)))   # las tres opciones conservan el signo (p1 < 0 < p99)

cat("Transformación aplicada:", TRANSF_CNMR, "\n")
cat(sprintf("Cortes p1–p99 de CNMR (1950–2023): [%.2f, %.2f]\n", cortes_cnmr[1], cortes_cnmr[2]))
rbind(CNMR = summary(panel$CNMR), CNMR_t = summary(panel$CNMR_t)) |> round(3) |> print()


Transformación aplicada: asinh 
Cortes p1–p99 de CNMR (1950–2023): [-38.63, 54.93]
           Min. 1st Qu. Median   Mean 3rd Qu.    Max.
CNMR   -709.899  -1.652 -0.225 -0.162   1.214 357.140
CNMR_t   -7.258  -1.276 -0.223 -0.160   1.025   6.571


## g) Guardado

- `panel_paises.rds`: panel largo 237 países × 151 años, con identificadores, región, subregión, `es_pequeno`, los indicadores de `IND_AMPLIADO`, `CNMR_t` y `TPopulation1July`.
- `meta_paises.rds`: una fila por país con su población en 2023.
- `matrices_1950_2023.rds` y `matrices_1950_2100.rds`: listas nombradas por indicador (`TFR`, `LEx`, `CNMR_t`, `MAC`, `MedianAgePop`) de matrices país × año, con filas = `ISO3_code` (ordenadas igual que `meta_paises`) y columnas = año. Incluyen los 237 países; el filtro de pequeños se aplica en cada fase con `es_pequeno`.


In [8]:
panel_paises <- panel |>
  inner_join(pop_ref |> select(LocID, es_pequeno), by = "LocID") |>
  select(LocID, ISO3_code, Location, region, subregion, es_pequeno, Time,
         all_of(IND_AMPLIADO), CNMR_t, TPopulation1July) |>
  arrange(LocID, Time)

meta_paises <- paises_geo |>
  inner_join(pop_ref, by = "LocID") |>
  select(LocID, ISO3_code, Location, region, subregion, es_pequeno, pop_2023) |>
  arrange(LocID)

IND_MATRIZ <- replace(IND_AMPLIADO, IND_AMPLIADO == "CNMR", "CNMR_t")

construir_matrices <- function(datos, horizonte) {
  d <- datos |> filter(Time >= horizonte[1], Time <= horizonte[2])
  anios <- horizonte[1]:horizonte[2]
  m <- lapply(set_names(IND_MATRIZ), function(ind) {
    w <- d |>
      select(ISO3_code, Time, valor = all_of(ind)) |>
      pivot_wider(names_from = Time, values_from = valor)
    x <- as.matrix(w[, as.character(anios)])
    rownames(x) <- w$ISO3_code
    x[meta_paises$ISO3_code, , drop = FALSE]
  })
  stopifnot(all(sapply(m, function(x) identical(dim(x), c(237L, length(anios))))),
            all(sapply(m, function(x) !anyNA(x))),
            all(sapply(m, function(x) identical(rownames(x), meta_paises$ISO3_code))))
  m
}

matrices_1950_2023 <- construir_matrices(panel_paises, HORIZONTE)
matrices_1950_2100 <- construir_matrices(panel_paises, HORIZONTE_SENS)

# Verificación puntual: un valor de la matriz coincide con el panel
stopifnot(matrices_1950_2023$TFR["COL", "1990"] ==
            panel_paises$TFR[panel_paises$ISO3_code == "COL" & panel_paises$Time == 1990])

saveRDS(panel_paises,       here::here("output/data/panel_paises.rds"))
saveRDS(meta_paises,        here::here("output/data/meta_paises.rds"))
saveRDS(matrices_1950_2023, here::here("output/data/matrices_1950_2023.rds"))
saveRDS(matrices_1950_2100, here::here("output/data/matrices_1950_2100.rds"))

cat("panel_paises:", dim(panel_paises), "\n")
cat("meta_paises: ", dim(meta_paises), "\n")
cat("matrices_1950_2023:", names(matrices_1950_2023), "→", dim(matrices_1950_2023[[1]]), "\n")
cat("matrices_1950_2100:", names(matrices_1950_2100), "→", dim(matrices_1950_2100[[1]]), "\n")


panel_paises: 35787 14 
meta_paises:  237 7 
matrices_1950_2023: TFR LEx CNMR_t MAC MedianAgePop → 237 74 
matrices_1950_2100: TFR LEx CNMR_t MAC MedianAgePop → 237 151 


## h) Resumen

Número de países (total, pequeños y no pequeños), conteo por región y rango de cada indicador en los dos horizontes, sobre los 237 países.


In [9]:
cat(sprintf("Países: %d en total | %d pequeños | %d no pequeños (análisis principal)\n\n",
            nrow(meta_paises), sum(meta_paises$es_pequeno), sum(!meta_paises$es_pequeno)))

resumen_region <- meta_paises |>
  group_by(region) |>
  summarise(total = n(), pequenos = sum(es_pequeno), no_pequenos = sum(!es_pequeno), .groups = "drop") |>
  arrange(desc(total))
print(as.data.frame(resumen_region), row.names = FALSE)

rango_ind <- function(datos, horizonte) {
  datos |>
    filter(Time >= horizonte[1], Time <= horizonte[2]) |>
    select(all_of(c(IND_AMPLIADO, "CNMR_t"))) |>
    pivot_longer(everything(), names_to = "indicador") |>
    group_by(indicador) |>
    summarise(min = min(value), max = max(value), .groups = "drop") |>
    rename_with(~ paste0(.x, "_", horizonte[1], "_", horizonte[2]), c(min, max))
}
resumen_rangos <- inner_join(rango_ind(panel_paises, HORIZONTE),
                             rango_ind(panel_paises, HORIZONTE_SENS), by = "indicador") |>
  slice(match(c(IND_AMPLIADO, "CNMR_t"), indicador)) |>
  mutate(across(where(is.numeric), ~ round(.x, 2)))
cat("\nRango de cada indicador (237 países):\n")
print(as.data.frame(resumen_rangos), row.names = FALSE)


Países: 237 en total | 37 pequeños | 200 no pequeños (análisis principal)

                          region total pequenos no_pequenos
                          Africa    58        1          57
                            Asia    51        0          51
                          Europe    50        9          41
 Latin America and the Caribbean    50       14          36
                         Oceania    23       10          13
                Northern America     5        3           2

Rango de cada indicador (237 países):
    indicador min_1950_2023 max_1950_2023 min_1950_2100 max_1950_2100
          TFR          0.66          8.86          0.66          8.86
          LEx         10.99         86.37         10.99         94.67
         CNMR       -709.90        357.14       -709.90        357.14
          MAC         23.69         33.42         23.69         34.99
 MedianAgePop         12.62         62.42         12.62         72.36
       CNMR_t         -7.26          6.57     

## Conclusiones de F1

- **Lectura:** el archivo Medium tiene 84,360 filas × 67 columnas (555 LocID × 152 años), con el BOM bien manejado. El suplemento tiene 35,636 filas (236 países × 151 años); no se usa en esta fase.
- **Jerarquía geográfica:** los 237 países tienen subregión (22) y región (6). Por región: Africa 58, Asia 51, Europe 50, Latin America and the Caribbean 50, Oceania 23 y Northern America 5. La regla 918 → 905 cubre Bermuda, Canada, Greenland, Saint Pierre and Miquelon y USA.
- **Panel:** 237 países × 151 años (1950–2100), sin NA en `TFR`, `LEx`, `CNMR`, `MAC`, `MedianAgePop` ni `TPopulation1July`.
- **Consistencia:** la suma de los países reproduce World con una diferencia máxima de 0.014 mil, así que el filtro es completo.
- **Países pequeños:** son 37 (< 100 mil en 2023), tal como se esperaba. Quedan **200 países** para el análisis principal.
- **CNMR:** tiene colas extremas en países que *no* son pequeños: Kuwait 1990 (−710 ‰) y 1991 (+341 ‰), Hong Kong 1964/1967, Macao 1964, Liberia 1990. Por eso la transformación es necesaria aunque se excluyan los pequeños. Con `asinh`, el rango pasa de [−710, 357] a [−7.3, 6.6].
- **LEx:** hay mínimos de crisis (South Sudan 1988: 11.0; Cambodia 1977–78: ~11). Son caídas reales y se revisarán como posibles outliers funcionales en F2.
- **Salidas:** `panel_paises.rds`, `meta_paises.rds`, `matrices_1950_2023.rds` (237 × 74) y `matrices_1950_2100.rds` (237 × 151), con filas = `ISO3_code` en el orden de `meta_paises`.
